# 🚀 Fine-Tune Venture-Coder (Qwen 14B) on Luau with Unsloth

This notebook fine-tunes `Qwen2.5-Coder-14B-Instruct` using **QLoRA** on a free Google Colab **T4 GPU (16 GB VRAM)**.

### What this accomplishes:
1. Teaches the model the **Roblox Senior Engineering Codex** (--!strict, Services pattern, session locking, Selene linting).
2. Trains on captured repair attempts (`data/training/repairs.jsonl`) so the model stops dropping tokens and syntax.
3. Exports directly to **GGUF format** for 1-click import into **Ollama** on your local machine.

## 1. Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" "trl<0.9.0" peft accelerate bitsandbytes datasets

## 2. Load Base Model (Qwen2.5-Coder-14B 4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None for auto-detection (Float16 on T4)
load_in_4bit = True  # Fits 14B comfortably inside 16GB VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 3. Upload and Prepare Dataset (`repairs.jsonl`)

In [ ]:
from google.colab import files
import os

print("Upload your repairs.jsonl file from data/training/repairs.jsonl:")
uploaded = files.upload()
dataset_path = list(uploaded.keys())[0]
print(f"Loaded {dataset_path}")

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = load_dataset("json", data_files=dataset_path, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

## 4. Train Model with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import transformers, gc, torch

# 1. Clean GPU memory
gc.collect()
torch.cuda.empty_cache()

# 2. Compatibility bridge for Trainer arguments
if not hasattr(transformers.Trainer, '_original_init'):
    transformers.Trainer._original_init = transformers.Trainer.__init__
    def _patched_init(self, *args, **kwargs):
        if 'tokenizer' in kwargs:
            kwargs['processing_class'] = kwargs.pop('tokenizer')
        return transformers.Trainer._original_init(self, *args, **kwargs)
    transformers.Trainer.__init__ = _patched_init

# 3. Fix PyTorch in-place loss view error (transformers v4.49+ in-place *= issue)
orig_forward = model.forward
def safe_forward(*args, **kwargs):
    res = orig_forward(*args, **kwargs)
    if hasattr(res, 'loss') and res.loss is not None:
        res.loss = res.loss.clone()
    elif isinstance(res, dict) and 'loss' in res and res['loss'] is not None:
        res['loss'] = res['loss'].clone()
    return res
model.forward = safe_forward

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()


## 5. Export directly to GGUF (for Ollama)

In [ ]:
# 1. Save complete raw LoRA adapter & tokenizer (~150 MB)
model.save_pretrained("venture_coder_lora")
tokenizer.save_pretrained("venture_coder_lora")
print("✅ Saved raw LoRA adapter to venture_coder_lora/")

# 2. Install gguf dependency & shallow clone llama.cpp conversion scripts
!pip install -q gguf
![ -d "llama.cpp" ] || git clone --depth 1 https://github.com/ggerganov/llama.cpp

# 3. Convert LoRA adapter to GGUF format
print("⏳ Converting LoRA adapter to venture_coder_adapter.gguf...")
!cd llama.cpp && python convert_lora_to_gguf.py ../venture_coder_lora --base-model-id Qwen/Qwen2.5-Coder-14B-Instruct --outtype f16 --outfile ../venture_coder_adapter.gguf
print("✅ Converted adapter to GGUF successfully!")

# 4. Package everything (both .gguf adapter + full raw LoRA directory) into one zip archive
!zip -q -r venture_coder_bundle.zip venture_coder_adapter.gguf venture_coder_lora
print("✅ Packaged full bundle into venture_coder_bundle.zip")

# 5. Download the complete bundle to your local PC
from google.colab import files
print("⬇️ Initiating download of venture_coder_bundle.zip... Check your browser downloads!")
files.download("venture_coder_bundle.zip")


## 6. How to Import into Local Ollama

Once `venture_coder_adapter.gguf` finishes downloading to your PC (e.g. in your Downloads or project folder):

1. Create a `Modelfile` in the same directory:
```text
FROM qwen2.5-coder:14b
ADAPTER ./venture_coder_adapter.gguf
PARAMETER temperature 0.1
PARAMETER top_p 0.95
```

2. Run in PowerShell:
```powershell
ollama create venture-coder:14b -f Modelfile
```

Ollama attaches your fine-tuned LoRA directly to your local `qwen2.5-coder:14b` in 2 seconds!